# GeoLifeCLEF 2025 — Baseline Tabulaire (Kaggle)

**Approche** : Features environnementales + LightGBM One-vs-Rest multi-label

## Instructions
1. Créer un nouveau notebook sur la page du challenge Kaggle
2. Coller ce code cellule par cellule (ou uploader ce .ipynb)
3. Dans le panneau de droite **Data** → vérifier que le dataset `geolifeclef-2025` et `geoplant` sont attachés
4. Activer GPU : **Session options > Accelerator > GPU T4**

In [ ]:
import os

# Chercher automatiquement les chemins Kaggle
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.csv') and 'PA' in f and 'metadata' in f:
            print(os.path.join(root, f))

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

# ─── Chemins Kaggle ──────────────────────────────────────────────────────────
DATA = Path('/kaggle/input/competitions/le-challenge-deep-learning-miashs-2026')
ENV  = DATA / 'EnvironmentalValues'

# Vérification
print('Fichiers racine:', list(DATA.glob('*.csv')))
print('Dossiers env:', list(ENV.iterdir()))

In [ ]:
# ─── 1. PA metadata ──────────────────────────────────────────────────────────
pa_train = pd.read_csv(DATA / 'GLC25_PA_metadata_train.csv')
pa_test  = pd.read_csv(DATA / 'GLC25_PA_metadata_test.csv')

train_surveys = pa_train[['surveyId','lon','lat','year','region']].drop_duplicates('surveyId')
test_surveys  = pa_test[['surveyId','lon','lat','year','region']].drop_duplicates('surveyId')

print(f'Train surveys: {len(train_surveys)} | Test surveys: {len(test_surveys)}')

train_labels = (
    pa_train.dropna(subset=['speciesId'])
    .groupby('surveyId')['speciesId']
    .apply(lambda x: list(x.astype(int).unique()))
    .reset_index()
    .rename(columns={'speciesId': 'species_list'})
)
print(f'Espèces uniques: {pa_train["speciesId"].nunique()}')

In [ ]:
# ─── 2. Features environnementales ───────────────────────────────────────────
def load_env(tr_path, te_path):
    return pd.read_csv(tr_path), pd.read_csv(te_path)

elev_tr, elev_te       = load_env(ENV/'Elevation/GLC25-PA-train-elevation.csv',
                                   ENV/'Elevation/GLC25-PA-test-elevation.csv')
soil_tr, soil_te       = load_env(ENV/'SoilGrids/GLC25-PA-train-soilgrids.csv',
                                   ENV/'SoilGrids/GLC25-PA-test-soilgrids.csv')
land_tr, land_te       = load_env(ENV/'LandCover/GLC25-PA-train-landcover.csv',
                                   ENV/'LandCover/GLC25-PA-test-landcover.csv')
foot_tr, foot_te       = load_env(ENV/'HumanFootprint/GLC25-PA-train-human_footprint.csv',
                                   ENV/'HumanFootprint/GLC25-PA-test-human_footprint.csv')
bioclim_tr, bioclim_te = load_env(ENV/'ClimateAverage_1981-2010/GLC25-PA-train-bioclimatic.csv',
                                   ENV/'ClimateAverage_1981-2010/GLC25-PA-test-bioclimatic.csv')

print('Shapes:', elev_tr.shape, soil_tr.shape, land_tr.shape, foot_tr.shape, bioclim_tr.shape)

In [ ]:
# ─── 3. Merge & preprocessing ────────────────────────────────────────────────
def merge_all(surveys, dfs):
    result = surveys.copy()
    for df in dfs:
        result = result.merge(df, on='surveyId', how='left')
    return result

X_train_df = merge_all(train_surveys, [elev_tr, soil_tr, land_tr, foot_tr, bioclim_tr])
X_test_df  = merge_all(test_surveys,  [elev_te, soil_te, land_te, foot_te, bioclim_te])

# Encoder la région
reg_tr = pd.get_dummies(X_train_df['region'], prefix='region')
reg_te = pd.get_dummies(X_test_df['region'],  prefix='region').reindex(columns=reg_tr.columns, fill_value=0)
X_train_df = pd.concat([X_train_df.drop(columns=['region']), reg_tr], axis=1)
X_test_df  = pd.concat([X_test_df.drop(columns=['region']),  reg_te], axis=1)

train_df = X_train_df.merge(train_labels, on='surveyId', how='inner')
feature_cols = [c for c in X_train_df.columns if c != 'surveyId']

X_train = train_df[feature_cols].values.astype(np.float32)
X_test  = X_test_df[feature_cols].values.astype(np.float32)

# Imputer NaN par médiane
col_medians = np.nanmedian(X_train, axis=0)
for i in range(X_train.shape[1]):
    X_train[np.isnan(X_train[:, i]), i] = col_medians[i]
    X_test[np.isnan(X_test[:, i]), i]   = col_medians[i]

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')

In [ ]:
# ─── 4. Encodage multi-label ─────────────────────────────────────────────────
all_species = sorted(pa_train['speciesId'].dropna().astype(int).unique())
mlb = MultiLabelBinarizer(classes=all_species)
Y_train = mlb.fit_transform(train_df['species_list'])
print(f'Y_train: {Y_train.shape}')

In [ ]:
# ─── 5. Entraînement LightGBM One-vs-Rest ────────────────────────────────────
species_counts = pa_train['speciesId'].dropna().astype(int).value_counts()
MIN_OCC = 10
common_species = species_counts[species_counts >= MIN_OCC].index.tolist()
species_to_idx = {s: i for i, s in enumerate(mlb.classes_)}
common_idx = [species_to_idx[s] for s in common_species if s in species_to_idx]
print(f'Espèces à entraîner: {len(common_species)}')

lgb_params = dict(
    objective='binary',
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=5,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    verbose=-1,
)

test_probs = np.zeros((len(X_test), len(all_species)), dtype=np.float32)

for i, (sp_idx, sp_id) in enumerate(zip(common_idx, common_species)):
    y = Y_train[:, sp_idx]
    if y.sum() < MIN_OCC:
        continue
    clf = LGBMClassifier(**lgb_params)
    clf.fit(X_train, y)
    test_probs[:, sp_idx] = clf.predict_proba(X_test)[:, 1]
    if (i + 1) % 200 == 0:
        print(f'  {i+1}/{len(common_idx)} espèces...')

print('Entraînement terminé!')

In [ ]:
# ─── 6. Génération soumission ─────────────────────────────────────────────────
THRESHOLD = 0.2
TOP_K = 10

predictions = []
for i in range(len(X_test)):
    probs = test_probs[i]
    above_thresh = np.where(probs >= THRESHOLD)[0]
    if len(above_thresh) < TOP_K:
        top_k_idx = np.argsort(probs)[::-1][:TOP_K]
        pred_idx = np.union1d(above_thresh, top_k_idx)
    else:
        pred_idx = above_thresh
    pred_species = sorted([int(mlb.classes_[j]) for j in pred_idx])
    predictions.append(' '.join(map(str, pred_species)))

submission = pd.DataFrame({
    'surveyId': X_test_df['surveyId'].values,
    'predictions': predictions
})

submission.to_csv('/kaggle/working/submission.csv', index=False)
print(f'Soumission sauvegardée! {len(submission)} surveys')
submission.head()